# Tipos de variables y visualización — Ejemplos

**Módulo 1 — Introducción · Curso Analítica de Datos**

Este notebook acompaña las diapositivas [`1.5_Tipos_de_Variables.pdf`](1.5_Tipos_de_Variables.pdf) y lleva a código lo que allí se explica: **cómo clasificar las variables de un dataset** (cuantitativas/cualitativas, discretas/continuas, nominales/ordinales) y **cómo elegir la visualización correcta** según ese tipo.

## Contenido

1. **Cargar un dataset de Kaggle**: el *Spotify Tracks Dataset*, con más de 100.000 canciones y sus características de audio.
2. **Identificación de tipos de variable**: clasificamos cada columna del dataset siguiendo el esquema de las diapositivas.
3. **Visualización según el tipo de variable**: un gráfico de barras (para una variable cualitativa) y un histograma (para una variable cuantitativa continua), tal como se explicó en las diapositivas.

> 💡 La sección 1 **requiere una cuenta de Kaggle y una clave de API** (gratis) — más abajo se explica paso a paso cómo obtenerla. Las secciones 2 y 3 no requieren nada adicional una vez el dataset está cargado.

In [ ]:
# Librerías que usaremos en todo el notebook
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (8, 5)

print('Librerías cargadas correctamente ✅')
print('pandas', pd.__version__)

---
## 1. Cargar un dataset de Kaggle: *Spotify Tracks Dataset*

[Kaggle](https://www.kaggle.com/) es una de las plataformas de datasets más populares (la mencionamos en las diapositivas de la unidad anterior). Vamos a usar el [**Spotify Tracks Dataset**](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset): más de 114.000 canciones con sus características de audio (qué tan bailable es, energía, tempo, si tiene voz o es instrumental, género, popularidad, etc.) — un dataset muy visual y cercano para explorar tipos de variables.

### Cómo obtener tu clave de API de Kaggle (una sola vez)

1. Crea una cuenta gratuita en [kaggle.com](https://www.kaggle.com/) si no tienes una.
2. Ve a tu perfil → **Settings** → sección **API** → botón **"Create New Token"**. Esto descarga un archivo `kaggle.json` con tus credenciales.
3. Instala la librería oficial de Kaggle (si no la tienes): `pip install kagglehub`
4. La primera vez que ejecutes la celda de descarga, `kagglehub` te pedirá autenticarte (puede abrir el navegador, o puedes colocar el archivo `kaggle.json` descargado en `~/.kaggle/kaggle.json`).

> Tu clave de API es personal — trátala como una contraseña y no la subas al repositorio.

In [ ]:
import kagglehub

# dataset_download descarga el dataset (o usa la copia en caché si ya lo habías
# descargado antes) y devuelve la ruta local donde quedaron los archivos.
try:
    ruta_dataset = kagglehub.dataset_download('maharshipandya/-spotify-tracks-dataset')
    print('Dataset descargado en:', ruta_dataset)
except Exception as error:
    print('❌ No se pudo descargar el dataset de Kaggle.')
    print('Verifica tu conexión a internet y que tu API key esté configurada (ver celda anterior).')
    print('Detalle del error:', error)
    raise

In [ ]:
import os

# El dataset puede traer más de un archivo; buscamos el primero que sea .csv
# en vez de asumir un nombre exacto (así el código no se rompe si Kaggle
# cambia el nombre del archivo en una futura versión del dataset).
archivos_csv = [f for f in os.listdir(ruta_dataset) if f.endswith('.csv')]
print('Archivos CSV encontrados:', archivos_csv)

ruta_csv = os.path.join(ruta_dataset, archivos_csv[0])
canciones = pd.read_csv(ruta_csv, index_col=0)  # la primera columna es un índice numérico de fila

print(f'\nDataset cargado: {canciones.shape[0]} filas x {canciones.shape[1]} columnas')
canciones.head()

In [ ]:
canciones.info()

---
## 2. Identificación de tipos de variable

Repasando la clasificación de las diapositivas:

- **Cuantitativas** (expresan cantidades): **discretas** (se cuentan, valores enteros separados) o **continuas** (se miden, cualquier valor dentro de un rango).
- **Cualitativas** (expresan categorías): **nominales** (sin orden) u **ordinales** (con jerarquía).

⚠️ **Cuidado**: que una columna esté almacenada como número **no la vuelve automáticamente cuantitativa**. Hay que fijarse en qué *significa* el valor, no solo en su tipo de dato — por eso este dataset trae un par de columnas "trampa" que vale la pena revisar con cuidado.

In [ ]:
# Clasificación de las columnas más relevantes del dataset, con su justificación.
# La armamos como un DataFrame para poder mostrarla como una tabla ordenada.
clasificacion = pd.DataFrame([
    ('track_genre',      'Cualitativa', 'Nominal',              'Categoría musical sin orden lógico entre géneros'),
    ('artists',          'Cualitativa', 'Nominal',              'Nombre del artista: una categoría, no una cantidad'),
    ('explicit',         'Cualitativa', 'Nominal (binaria)',    'Solo dos categorías: contenido explícito o no'),
    ('key',               'Cualitativa', 'Nominal',              '⚠️ Se guarda como número (0-11), pero representa una nota musical (Do, Do#, Re...), no una cantidad'),
    ('mode',              'Cualitativa', 'Nominal (binaria)',    '⚠️ También numérica en apariencia: 0 = modo menor, 1 = modo mayor — son categorías, no cantidades'),
    ('popularity',        'Cuantitativa', 'Discreta',            'Puntaje entero de 0 a 100; se cuenta en unidades enteras'),
    ('duration_ms',       'Cuantitativa', 'Continua',            'Duración de la canción; el tiempo es una magnitud medible, no contable'),
    ('tempo',             'Cuantitativa', 'Continua',            'Pulsaciones por minuto (BPM); puede tomar cualquier valor decimal'),
    ('loudness',          'Cuantitativa', 'Continua',            'Volumen promedio en decibeles (dB)'),
    ('danceability',      'Cuantitativa', 'Continua',            'Índice entre 0.0 y 1.0 que mide qué tan bailable es la canción'),
    ('energy',            'Cuantitativa', 'Continua',            'Índice entre 0.0 y 1.0 de intensidad y actividad percibida'),
], columns=['columna', 'tipo_general', 'subtipo', 'justificacion'])

clasificacion

In [ ]:
# Verifiquemos con código un par de las afirmaciones de la tabla anterior.

# 1) 'popularity' es discreta: todos sus valores deberían ser números enteros
es_entero = (canciones['popularity'].dropna() % 1 == 0).all()
print(f"¿'popularity' toma solo valores enteros? {es_entero}")

# 2) 'key' parece numérica, pero en realidad solo toma 12 valores posibles (0 a 11):
#    uno por cada nota de la escala musical — típico de una variable nominal, no continua.
print(f"\nValores únicos de 'key': {sorted(canciones['key'].unique())}")

# 3) 'track_genre' es nominal: veamos cuántas categorías distintas tiene (sin ningún orden entre ellas)
print(f"\n'track_genre' tiene {canciones['track_genre'].nunique()} categorías distintas, por ejemplo:")
print(canciones['track_genre'].unique()[:10])

---
## 3. Visualización según el tipo de variable

Tal como se explicó en las diapositivas:

- Para una **variable cualitativa** (categorías), usamos un **gráfico de barras**: cada barra es una categoría y su longitud es la frecuencia.
- Para una **variable cuantitativa continua**, usamos un **histograma**: agrupa los valores en intervalos para mostrar cómo se distribuyen.

### Gráfico de barras — frecuencia por género musical (`track_genre`, cualitativa nominal)

In [ ]:
# Tomamos los 10 géneros más frecuentes para que la gráfica sea legible
# (el dataset completo tiene más de 100 géneros distintos)
frecuencia_generos = canciones['track_genre'].value_counts().head(10)

fig, ax = plt.subplots()
ax.bar(frecuencia_generos.index, frecuencia_generos.values, color='#1DB954')  # verde Spotify

ax.set_title('Top 10 géneros musicales más frecuentes en el dataset')
ax.set_xlabel('Género')
ax.set_ylabel('Cantidad de canciones')
ax.tick_params(axis='x', rotation=40)
plt.tight_layout()
plt.show()

print(f"Género más frecuente: '{frecuencia_generos.idxmax()}' con {frecuencia_generos.max()} canciones.")

### Histograma — distribución del tempo (`tempo`, cuantitativa continua)

In [ ]:
fig, ax = plt.subplots()

# 30 intervalos (bins): un punto intermedio razonable, ni muy agregado ni muy ruidoso
ax.hist(canciones['tempo'].dropna(), bins=30, color='#1DB954', edgecolor='white')

ax.set_title('Distribución del tempo de las canciones (BPM)')
ax.set_xlabel('Tempo (beats por minuto)')
ax.set_ylabel('Cantidad de canciones')
plt.tight_layout()
plt.show()

print('Tempo promedio:', round(canciones['tempo'].mean(), 1), 'BPM')
print('Tempo más común (mediana):', round(canciones['tempo'].median(), 1), 'BPM')

---
## Cierre

En este notebook vimos, en código, las ideas de las diapositivas [`1.5_Tipos_de_Variables.pdf`](1.5_Tipos_de_Variables.pdf):

- Cómo cargar un dataset real desde **Kaggle** usando `kagglehub`.
- Cómo **clasificar cada variable** de un dataset (cuantitativa/cualitativa, discreta/continua, nominal/ordinal) — y cómo detectar el caso "trampa" de columnas numéricas que en realidad son categóricas (`key`, `mode`).
- Cómo elegir la **visualización correcta** según el tipo de variable: gráfico de barras para categorías, histograma para variables continuas.

### Recursos adicionales
- [Spotify Tracks Dataset en Kaggle](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)
- [Documentación de `kagglehub`](https://github.com/Kaggle/kagglehub)
- [Documentación de `matplotlib.pyplot.hist`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html)